# FinPulse: Exploratory Data Analysis & Diagnostic Econometrics
This notebook performs exploratory analysis, visualization, and diagnostic testing on the processed SPY and VIX financial time series (2021-2025).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

%matplotlib inline
os.makedirs('figures', exist_ok=True)

## 1. Load Dataset

In [ ]:
data = pd.read_csv('FinPulse_Processed_Data.csv', parse_dates=['Date'])
print('Total dataset shape:', data.shape)
data.head()

## 2. Descriptive Summary Statistics

In [ ]:
stat_cols = ['Close', 'Return', 'Abs_Return', 'VIX', 'Parkinson']
data[stat_cols].describe().round(4)

## 3. Visualizations
### SPY Price, Returns, Absolute Returns, and VIX

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
axes[0].plot(data['Date'], data['Close'], color='#1E3A8A')
axes[0].set_title('SPY Closing Price ($)', fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(data['Date'], data['Return'], color='#2563EB', alpha=0.8)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('SPY Daily Log Returns (%)', fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.6)

axes[2].plot(data['Date'], data['Abs_Return'], color='#D97706', alpha=0.8)
axes[2].set_title('SPY Daily Absolute Returns (Abs_Return %)', fontweight='bold')
axes[2].grid(True, linestyle=':', alpha=0.6)

axes[3].plot(data['Date'], data['VIX'], color='#7C3AED')
axes[3].axhline(20, color='#EAB308', linestyle='--', label='VIX=20 Threshold')
axes[3].axhline(30, color='#EF4444', linestyle='-.', label='VIX=30 Threshold')
axes[3].set_title('CBOE Volatility Index (VIX)', fontweight='bold')
axes[3].grid(True, linestyle=':', alpha=0.6)
axes[3].legend()
plt.tight_layout()
plt.show()

### Return Distribution & Leptokurtosis (Q-Q Plot)

In [ ]:
ret = data['Return'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mu, std = ret.mean(), ret.std()
x = np.linspace(ret.min(), ret.max(), 300)
axes[0].hist(ret, bins=60, density=True, alpha=0.65, color='#3B82F6', edgecolor='#1D4ED8')
axes[0].plot(x, stats.norm.pdf(x, mu, std), color='#DC2626', linewidth=2)
axes[0].set_title('Distribution of Returns vs Normal', fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.6)

stats.probplot(ret, dist='norm', plot=axes[1])
axes[1].get_lines()[0].set_markerfacecolor('#2563EB')
axes[1].get_lines()[1].set_color('#DC2626')
axes[1].set_title('Normal Q-Q Plot (Fat Tails)', fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### Autocorrelation (ACF) & Volatility Clustering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
nlags = 30
lags = np.arange(nlags + 1)
conf_bound = 1.96 / np.sqrt(len(ret))
axes[0].stem(lags, acf(ret, nlags=nlags))
axes[0].axhline(conf_bound, color='red', linestyle='--')
axes[0].axhline(-conf_bound, color='red', linestyle='--')
axes[0].set_title('SPY Returns ACF (No Linear Dependence)', fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].stem(lags, acf(data['Abs_Return'], nlags=nlags))
axes[1].axhline(conf_bound, color='red', linestyle='--')
axes[1].axhline(-conf_bound, color='red', linestyle='--')
axes[1].set_title('SPY Absolute Returns ACF (Volatility Clustering)', fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Formal Econometric Diagnostics

In [ ]:
print('=== ADF TEST ===')
adf_close = adfuller(data['Close'].dropna())
adf_ret = adfuller(data['Return'].dropna())
print(f'Close: ADF Stat = {adf_close[0]:.4f}, p-value = {adf_close[1]:.4f}')
print(f'Return: ADF Stat = {adf_ret[0]:.4f}, p-value = {adf_ret[1]:.4f}')

print('\n=== LJUNG-BOX TEST ===')
print('Returns:')
print(acorr_ljungbox(data['Return'].dropna(), lags=[5, 10, 20], return_df=True))
print('\nAbsolute Returns:')
print(acorr_ljungbox(data['Abs_Return'].dropna(), lags=[5, 10, 20], return_df=True))

print('\n=== ENGLE ARCH-LM TEST ===')
arch_res = het_arch(data['Return'].dropna())
print(f'LM Stat: {arch_res[0]:.4f}, p-val: {arch_res[1]:.4e}, F-Stat: {arch_res[2]:.4f}, F p-val: {arch_res[3]:.4e}')

print('\n=== JARQUE-BERA TEST ===')
jb_ret = stats.jarque_bera(data['Return'].dropna())
print(f'Returns: JB Stat = {jb_ret.statistic:.4f}, p-val = {jb_ret.pvalue:.4e}')